# OpsLM — QLoRA fine-tune of Qwen3-4B (Kaggle T4)

Kaggle twin of `training/notebooks/opslm_qlora_colab.ipynb` — same
`training/scripts/train_opslm_qlora.py` underneath, driven headlessly via
`kaggle kernels push` (see `training/kaggle/README.md`).

Requirements on the Kaggle account (one-time, in the web UI):
- **Phone-verified** account (GPU + internet access in kernels).
- Notebook editor → Add-ons → **Secrets** → add `HF_TOKEN` = a Hugging Face
  token with write access, and attach it to this notebook.

Session limit is 12h (run needs ~3–4h). If it dies anyway, push again with
`RESUME = True` below — checkpoints go to the Hub every 50 steps.

In [ ]:
import os

HF_USER = "dhf1234"                    # HF username (not the Kaggle one)
PUSH_REPO = f"{HF_USER}/OpsLM-v1"
RESUME = False                          # True when re-running after a dead session

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # single-GPU train on the T4 x2 box

!nvidia-smi -L
print("will push to:", PUSH_REPO)

In [ ]:
!pip -q install "unsloth==2025.*" "trl>=0.12,<0.20" "datasets>=3" "huggingface_hub>=0.25"

The 534/59 train/val split is committed to the repo at
`data/sft/{train,val}.jsonl` — the clone already contains it.

In [ ]:
%cd /kaggle/working
!rm -rf OPsVerse
!git clone --depth 1 https://github.com/dilipna/OPsVerse.git
%cd OPsVerse

assert os.path.exists("data/sft/train.jsonl"), "data/sft missing from clone - check the repo"
print(sum(1 for _ in open("data/sft/train.jsonl")), "train examples")

In [ ]:
extra = "--resume" if RESUME else ""
!python training/scripts/train_opslm_qlora.py \
    --data-repo data/sft \
    --push-repo {PUSH_REPO} \
    --epochs 3 --save-steps 50 {extra}